In [ ]:
#将下载的多时间段nc文件进行拼接，不是所有的都需要拼接
import os
import xarray as xr

# 定义文件夹路径
folder1 = r'xx\xxx'  # 包含原始.nc文件的文件夹
folder2 = r'xx\xxxxx'  # 保存拼接后的.nc文件的文件夹

# 创建folder2（若没有）
if not os.path.exists(folder2):
    os.makedirs(folder2)

# 获取folder1下所有的.nc文件
nc_files = [os.path.join(folder1, f) for f in os.listdir(folder1) if f.endswith('.nc')]

# 使用xarray打开所有.nc文件并按时间维度拼接
try:
    ds = xr.open_mfdataset(nc_files, combine='by_coords')
except Exception as e:
    raise ValueError(f"文件拼接失败，请检查文件内容是否一致或时间维度是否正确")

# 定义输出文件路径
output_path = os.path.join(folder2, 'xx\xxx.nc')   # ！！！！！修改输出文件名称和路径

# 保存拼接后的数据集到新的.nc文件
ds.to_netcdf(output_path)

print(f"拼接后的文件已保存到: {output_path}")

In [ ]:
#(只用于处理降水pr)月相加计算年，原文件为一个nc文件（包含所有年的数据），处理后每年一个nc文件
import os
import xarray as xr
import numpy as np

# 定义输入文件路径和输出文件目录
input_file = r"xxx\\pr.nc"
output_dir = r"xxx\\pr"

# 创建输出目录（如果不存在）
os.makedirs(output_dir, exist_ok=True)

# 打开NC文件
ds = xr.open_dataset(input_file)

# 确保文件中有'time'维度和'pr'变量
if "time" not in ds.dims or "pr" not in ds.variables:
    raise ValueError("输入文件中缺少'time'维度或'pr'变量")

# 将时间转换为年份和月份
ds['year'] = ds['time'].dt.year
ds['month'] = ds['time'].dt.month

# 定义一个函数，用于计算每个月的天数
def get_days_in_month(year, month):
    if month == 2:  # 处理闰年
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            return 29
        else:
            return 28
    elif month in [4, 6, 9, 11]:  # 30天的月份
        return 30
    else:  # 31天的月份
        return 31

# 计算每年的总降水量
for year in range(2015, 2101):  # 2015年到2100年(2015,2101)
#for year in range(1985, 2015):  # 1985年到2014年(1985,2015)    
    # 筛选当前年份的数据
    yearly_data = ds['pr'].sel(time=ds['year'] == year)

    # 初始化年总降水量
    annual_sum = xr.zeros_like(yearly_data.isel(time=0))

    # 逐月计算降水量并累加
    for month in range(1, 13):  # 1月到12月
        # 筛选当前月份的数据
        monthly_data = yearly_data.sel(time=yearly_data['time'].dt.month == month)
        if len(monthly_data) > 0:  
            days_in_month = get_days_in_month(year, month)
            # 单位换算：kg m-2 s-1 -> mm
            # 1 kg m-2 s-1 = 1 mm/s，因此需要乘以秒数（86400秒/天）和当月天数
            monthly_mm = monthly_data * 86400 * days_in_month
            # 累加到年总降水量
            annual_sum += monthly_mm.sum(dim='time')

    annual_ds = xr.Dataset(
        {"pr": annual_sum},
        attrs=ds.attrs
    )

    # 更新变量的units属性为"mm"
    annual_ds['pr'].attrs['units'] = 'mm'

    # 为新的数据集添加坐标信息
    for coord in ds.coords:
        if coord != 'time':  # 排除time维度
            annual_ds[coord] = ds[coord]

    # 保存到新的NC文件
    output_file = os.path.join(output_dir, f"pr_{year}.nc")
    annual_ds.to_netcdf(output_file)
    print(f"保存 {year} 的总和到文件 {output_file}")

print("处理完成！")

In [ ]:
#(除降水pr外的其他变量)月平均计算每的平均值年，原文件为一个nc文件（所有年），处理后每年一个nc文件，注意修改变量名称×6
import os
import xarray as xr

# 定义输入文件路径和输出文件目录
input_file = r"xxx\\xxx.nc"  #输入路径
output_dir = r"xxx\\xxx"  # 输出目录

# 创建输出目录（如果不存在）
os.makedirs(output_dir, exist_ok=True)

# 打开NC文件
ds = xr.open_dataset(input_file)

# 确保文件中有 'time' 维度和 'vvv' 变量
if "time" not in ds.dims or "vvv" not in ds.variables:    # !!!!!!!!!!!!此处修改变量名称
    raise ValueError("输入文件中缺少 'time' 维度或 'vvv' 变量")

# 计算年均值
annual_ds = ds.resample(time="Y").mean()

# **手动复制变量的元数据**
annual_ds["vvv"].attrs = ds["vvv"].attrs  # !!!!!!!!!!!!此处修改变量名称×2
annual_ds.attrs = ds.attrs

# 遍历每年的数据并保存为独立的 nc 文件
for year in annual_ds.time.dt.year.values:
    yearly_data = annual_ds.sel(time=str(year))  # 选择某一年的数据
    yearly_data = yearly_data.assign_coords(time=[year])  # 仅保留年份信息

    # **确保变量属性不丢失**
    yearly_data["vvv"].attrs = ds["vvv"].attrs   # !!!!!!!!!!!!此处修改变量名称×2

    # 保存文件
    output_file = os.path.join(output_dir, f"vvv_{year}.nc")    # !!!!!!!!!!!!此处修改变量名称
    yearly_data.to_netcdf(output_file)
    print(f"保存 {year} 的年平均值到文件 {output_file}")

print("处理完成！")

In [ ]:
#将文件夹下的所有nc文件的分辨率统一为8640, 4320
import os
import glob
import xarray as xr
import numpy as np

# 定义文件夹路径
input_folder = r'xxx\\pet'
output_folder = r'xxx\\pet'

# 创建输出目录
os.makedirs(output_folder, exist_ok=True)

# 目标网格
target_lon_count, target_lat_count = 8640, 4320

# 计算目标经纬度
new_lon = np.linspace(-180, 180, target_lon_count + 1)[:-1]
new_lat = np.linspace(-90, 90, target_lat_count + 1)[:-1]

# 获取所有 .nc 文件
nc_files = glob.glob(os.path.join(input_folder, '*.nc'))

# 处理所有文件
for nc_file in nc_files:
    ds = xr.open_dataset(nc_file)

    # 确保 lon 在 [-180, 180] 范围
    ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby('lon')

    # 处理 lon=0 的情况
    if 0 not in ds.lon.values:
        ds_zero = ds.interp(lon=0, method="linear")  # 线性插值
    else:
        ds_zero = ds.sel(lon=0).copy(deep=True)

    ds_zero = ds_zero.assign_coords(lon=360)
    ds_ext = xr.concat([ds, ds_zero], dim="lon")

    # 插值，这里的变量是pet
    new_data = ds_ext.pet.interp(lat=new_lat, lon=new_lon, method='linear')

    # 保存处理后的数据
    filename = os.path.basename(nc_file)
    output_path = os.path.join(output_folder, filename)
    new_data.to_netcdf(output_path)

    ds.close()

print("批量处理完成！")

In [ ]:
#(除sfcWind、tas)计算7个模式的平均值
import os
import xarray as xr
import numpy as np
from tqdm import tqdm  # 用于显示进度条

# 定义输入文件夹路径和输出文件夹路径
input_folder = r"xxx\\xxx"
output_folder = r"xxx\\xxx"

# 创建输出文件夹（如果不存在）
os.makedirs(output_folder, exist_ok=True)

# 获取所有子文件夹
sub_folders = [os.path.join(input_folder, sub) for sub in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, sub))]

# 获取第一个子文件夹中的文件列表（所有子文件夹中的变量文件名称相同）
file_names = [f for f in os.listdir(sub_folders[0]) if f.endswith('.nc')]

# 遍历每个文件
for file_name in tqdm(file_names, desc="Processing files"):
    data_list = []

    # 遍历每个子文件夹，读取相同名称的文件
    for sub_folder in sub_folders:
        file_path = os.path.join(sub_folder, file_name)
        if os.path.exists(file_path):
            ds = xr.open_dataset(file_path)
            data_list.append(ds)

    if data_list:
        reference_ds = data_list[0]

        # **计算多模式平均值**
        combined_data = xr.concat(data_list, dim="model")
        mean_data = combined_data.mean(dim="model")

        # **补充缺失的属性信息**
        mean_data.attrs = reference_ds.attrs
        for var in mean_data.data_vars:
            if var in reference_ds:
                mean_data[var].attrs = reference_ds[var].attrs

        # 保存到输出文件夹
        output_path = os.path.join(output_folder, file_name)
        mean_data.to_netcdf(output_path)
        print(f"保存 {file_name} 的平均值到 {output_path}")

print("处理完成！")


In [ ]:
#(sfcWind、tas)计算7个模式的平均值
import os
import xarray as xr
import numpy as np
from tqdm import tqdm  # 用于显示进度条

# 定义输入文件夹路径和输出文件夹路径
input_folder = r"xxx\\xxx"
output_folder = r"xxx\\xxx"

# 创建输出文件夹（如果不存在）
os.makedirs(output_folder, exist_ok=True)

# 获取所有子文件夹
sub_folders = [os.path.join(input_folder, sub) for sub in os.listdir(input_folder) if os.path.isdir(os.path.join(input_folder, sub))]

# 获取第一个子文件夹中的文件列表（所有子文件夹中的变量文件名称相同）
file_names = [f for f in os.listdir(sub_folders[0]) if f.endswith('.nc')]

# 遍历每个文件
for file_name in tqdm(file_names, desc="Processing files"):
    data_list = []

    # 遍历每个子文件夹，读取相同名称的文件
    for sub_folder in sub_folders:
        file_path = os.path.join(sub_folder, file_name)
        if os.path.exists(file_path):
            ds = xr.open_dataset(file_path)
            data_list.append(ds)

    # 如果找到至少一个文件
    if data_list:
        # 使用 xarray 的 concat 和 mean 计算平均值
        combined_data = xr.concat(data_list, dim='height')
        mean_data = combined_data.mean(dim='height')

        # **补充缺失的属性信息**
        mean_data.attrs = reference_ds.attrs
        for var in mean_data.data_vars:
            if var in reference_ds:
                mean_data[var].attrs = reference_ds[var].attrs
        # 保存结果到输出文件夹
        output_path = os.path.join(output_folder, file_name)
        mean_data.to_netcdf(output_path)
        print(f"保存 {file_name} 的平均值到 {output_path}")

print("处理完成！")